In [2]:
import pandas as pd

df = pd.read_csv("../data/Processed Data/olympics_gdp_merged.csv") 

In [3]:
# using the same period used in the eda before, and counting up all the unique medal winning results
df = df[(df["Year"] >= 1960) & (df["Year"] <= 2016)].copy()

medalWon = df[df["Medal_Won"] == True].copy()
medal_unique = (
    medalWon.groupby(["Year", "Country_Name", "Country_Code", "Sport", "Event", "Medal"])
    .size()
    .reset_index(name="Count")
)

medalCounts = (
    medal_unique.groupby(["Country_Name", "Country_Code"])
    .size()
    .reset_index(name="total_medals")
)


In [4]:
# created a country gdp based data and merging this into one singular summary table
countryLvlGDP = (
    df.groupby(["Country_Name", "Country_Code"])
    .agg(
        avg_gdp_per_capita=("GDP_per_capita", "mean"),
        athlete_count=("Athlete_ID", "nunique")
    )
    .reset_index()
)

summary_countryGDP = countryLvlGDP.merge(
    medalCounts,
    on=["Country_Name", "Country_Code"],
    how="left"
)

In [5]:
#countries that have no medals will be 0 and created a clean colunm names for d3 later
summary_countryGDP["total_medals"] = summary_countryGDP["total_medals"].fillna(0).astype(int)

summary_countryGDP = summary_countryGDP.rename(columns={
    "Country_Name": "country",
    "Country_Code": "iso3"
})

In [6]:
#rows that have missing gdp were removed + reduced the clutter for the bubble chart
summary_countryGDP = summary_countryGDP.dropna(subset=["avg_gdp_per_capita"]).copy()

# summary_countryGDP = summary_countryGDP[summary_countryGDP["athlete_count"] >= 100].copy()

In [7]:
# saved the results so it can be reused
summary_countryGDP.to_csv("../data/summary_countryGDP.csv", index=False)

print(summary_countryGDP.head())
print("\nRows exported:", len(summary_countryGDP))
print("\nSaved to: ../data/summary_countryGDP.csv")

       country iso3  avg_gdp_per_capita  athlete_count  total_medals
0  Afghanistan  AFG          450.583371             57             2
1      Albania  ALB         2838.814175             45             0
4      Andorra  AND        25919.174804             61             0
7    Argentina  ARG         5822.056368           1329            36
8      Armenia  ARM         2079.603893            146            16

Rows exported: 125

Saved to: ../data/summary_countryGDP.csv
